In [34]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# directory that contains agent_tools.py (must run before importing agent_tools)
_pkg = Path.cwd()
# print(_pkg)
if (_pkg / "agent_tools.py").is_file():
    pkg = _pkg
else:
    pkg = _pkg.parent  # typical: cwd is .../notebook, module is one level up

sys.path.insert(0, str(pkg.resolve()))

# Drop stale cache so edits to agent_tools.py are picked up without restarting the kernel
sys.modules.pop("agent_tools", None)
import agent_tools as at
from agent_tools import AgentMemory


In [35]:
df = pd.read_parquet("..\data\heloc_dataset_v1.parquet")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\arifn\AppData\Local\Temp\ipykernel_20944\4115570229.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_parquet("..\data\heloc_dataset_v1.parquet")


In [36]:
df.head()

,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,...,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance,label,date
0,Bad,55,144,4,84,20,3,0,83,2,...,0,0,33,-8,8,1,1,69,0,2015-01-01
1,Bad,61,58,15,41,2,4,4,100,-7,...,0,0,0,-8,0,-8,-8,0,0,2015-01-01
2,Bad,67,66,5,24,9,0,0,100,-7,...,4,4,53,66,4,2,1,86,0,2015-01-01
3,Bad,66,169,1,73,28,1,1,93,76,...,5,4,72,83,6,4,3,91,0,2015-01-01
4,Bad,81,333,27,132,12,0,0,100,-7,...,1,1,51,89,3,1,0,80,0,2015-01-01


# Metadata

In [37]:
df.columns

Index(['RiskPerformance', 'ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance', 'label',
       'date'],
      dtype='str')

In [38]:
am = AgentMemory()

In [39]:
am.col_time = 'date'
am.col_target = 'label'
am.cols_feat = ['ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance']
am.col_day = 'day'
am.col_month = 'month'
am.col_type = 'type'
am.col_score = 'score'

In [40]:
df[am.col_day] = at.get_day_from_date(df, am.col_time)
df[am.col_month] = at.get_month_from_date(df, am.col_time)
am.cols_feat_num, am.cols_feat_Cat, am.cols_feat_time = at.get_feature_dtype(df, am.cols_feat)


# EDA

In [41]:
am.df_nan_rate = at.get_nan_rate(df, am.cols_feat)
display(am.df_nan_rate)
am.df_nan_rate_timely = at.get_nan_rate_timely(df, am.cols_feat, am.col_month)
display(am.df_nan_rate_timely)

,feature_name,missing_rate
0,ExternalRiskEstimate,0.0
1,MSinceOldestTradeOpen,0.0
2,MSinceMostRecentTradeOpen,0.0
3,AverageMInFile,0.0
4,NumSatisfactoryTrades,0.0
5,NumTrades60Ever2DerogPubRec,0.0
6,NumTrades90Ever2DerogPubRec,0.0
7,PercentTradesNeverDelq,0.0
8,MSinceMostRecentDelq,0.0
9,MaxDelq2PublicRecLast12M,0.0


missing_rate                              \
time_period                              201501 201502 201503 201504 201505   
feature_name                                                                  
AverageMInFile                              0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days                0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen                   0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                       0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M                    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                                 0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden                    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden                  0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization          0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                                0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                       0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance                    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance                  0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                       0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                              0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                      0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                        0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                      0.0    0.0    0.0    0.0    0.0   
PercentTradesWBalance                       0.0    0.0    0.0    0.0    0.0   

                                                                              \
time_period                        201506 201507 201508 201509 201510 201511   
feature_name                                                                   
AverageMInFile                        0.0    0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days          0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen             0.0    0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                 0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M              0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                           0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden              0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden            0.0    0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization    0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                          0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                 0.0    0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance              0.0    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance            0.0    0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                 0.0    0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                        0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                0.0    0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                  0.0    0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                0.0    0.0    0.0    0

In [42]:
am.df_target_rate_timely = at.get_timely_binary_target_rate(df, am.col_month, am.col_target)

In [43]:
am.df_target_rate_timely

,mean,count
month,,
201501,0.480534,899
201502,0.396552,812
201503,0.353726,899
201504,0.480460,870
201505,0.583982,899
201506,0.489655,870
201507,0.474972,899
201508,0.527374,895
201509,0.386905,840


# Split Data

In [44]:
am.oot_th = 201510
am.hoot_th = 201503

In [45]:
# Use the same integer scale for `col_period` and for oot_th / hoot_th (here: `month`).
df[am.col_type] = at.split_data(df, am.col_target, am.col_month, am.oot_th, am.hoot_th)

In [46]:
df[am.col_type].value_counts()

type
train    2636
hoot     2610
oot      2576
test     1319
valid    1318
Name: count, dtype: int64

In [47]:
am.df_target_rate_per_sample = at.get_target_rate_sample(df, am.col_type, am.col_target)
display(am.df_target_rate_per_sample)


,count,mean
type,,
hoot,2610,0.410728
oot,2576,0.518245
test,1319,0.492039
train,2636,0.491654
valid,1318,0.491654


# Feature transformation

In [56]:
am.dict_bin, am.bp, am.df_binning_stats = at.get_optimal_bin(df.loc[df[am.col_type] == 'train'], am.cols_feat, am.col_target, cols_feat_cat=am.cols_feat_cat)

In [57]:
display(am.df_binning_stats.sort_values(by='gini_power', ascending=False))

,name,dtype,status,selected,n_bins,iv,js,gini,quality_score,gini_power
11,NumTotalTrades,numerical,OPTIMAL,True,6,0.024596,0.003058,0.079225,0.002017,0.920775
2,MSinceMostRecentTradeOpen,numerical,OPTIMAL,True,6,0.035198,0.004382,0.100193,0.018778,0.899807
20,NumInstallTradesWBalance,numerical,OPTIMAL,True,6,0.042386,0.005271,0.108321,0.00329,0.891679
12,NumTradesOpeninLast12M,numerical,OPTIMAL,True,6,0.048713,0.006021,0.111746,0.010252,0.888254
4,NumSatisfactoryTrades,numerical,OPTIMAL,True,6,0.074214,0.009178,0.136911,0.064841,0.863089
6,NumTrades90Ever2DerogPubRec,numerical,OPTIMAL,True,3,0.126368,0.01552,0.143345,0.148225,0.856655
18,NetFractionInstallBurden,numerical,OPTIMAL,True,6,0.070341,0.008731,0.14392,0.031213,0.85608
19,NumRevolvingTradesWBalance,numerical,OPTIMAL,True,6,0.086865,0.010744,0.158586,0.160678,0.841414
16,NumInqLast6Mexcl7days,numerical,OPTIMAL,True,5,0.113001,0.013863,0.168369,0.106171,0.831631
15,NumInqLast6M,numerical,OPTIMAL,True,5,0.129148,0.015784,0.180347,0.059235,0.819653


In [58]:
am.cols_feat_woe = [col+'_woe' for col in am.cols_feat]

In [60]:
X_train_woe = am.bp.transform(df[am.cols_feat], metric="woe")

In [64]:
X_train_woe


,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,1.212731,0.026363,0.068578,-0.308196,-0.176618,0.982534,-0.179943,0.980232,1.145268,0.959119,...,0.077839,0.431781,-0.336157,-0.293976,0.204033,-0.212588,0.638483,-0.083471,0.254295,0.133993
1,1.212731,1.022666,-0.132759,0.793657,0.675509,0.982534,0.818584,-0.459261,-0.478819,0.283709,...,0.903738,0.431781,-0.336157,-0.293976,-0.723471,-0.212588,-0.017700,-0.415595,-0.423027,-0.834669
2,0.701260,1.022666,0.068578,0.793657,0.432487,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,0.077839,0.431781,0.425731,0.477439,0.429050,0.193523,-0.147205,-0.043969,0.254295,0.641411
3,0.701260,0.026363,0.107014,-0.051872,-0.176618,0.515046,0.634442,0.202690,-0.371713,0.024784,...,0.586148,0.431781,1.011981,0.477439,1.235124,0.226612,0.173665,0.324450,0.760856,1.109830
4,-1.352153,-0.669376,-0.357437,-0.639523,-0.012768,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,-0.136823,0.431781,0.038934,0.011598,0.429050,0.226612,-0.147205,-0.083471,-0.423027,0.361089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10454,0.019411,0.526229,0.068578,0.462762,-0.176618,-0.266698,-0.179943,0.202690,-0.371713,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,0.173665,-0.043969,-0.423027,1.109830
10455,0.701260,0.026363,-0.357437,-0.051872,0.272780,-0.266698,-0.179943,0.202690,0.089215,0.024784,...,0.047152,-0.044198,0.038934,0.011598,1.495328,-0.048350,-0.390585,-0.043969,0.254295,0.361089
10456,0.019411,0.526229,0.068578,0.462762,-0.012768,0.515046,0.634442,-0.459261,-0.478819,0.024784,...,-0.136823,-0.456448,0.425731,0.477439,-0.723471,-0.212588,0.173665,-0.415595,-0.423027,-0.347168
10457,0.019411,-0.217391,-0.132759,-0.639523,-0.365827,0.802181,0.818584,-0.459261,0.089215,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,-0.147205,-0.083471,-0.423027,-0.834669


In [65]:
df[am.cols_feat_woe] = am.bp.transform(df[am.cols_feat], metric="woe")

In [66]:
df[am.cols_feat_woe]

,ExternalRiskEstimate_woe,MSinceOldestTradeOpen_woe,MSinceMostRecentTradeOpen_woe,AverageMInFile_woe,NumSatisfactoryTrades_woe,NumTrades60Ever2DerogPubRec_woe,NumTrades90Ever2DerogPubRec_woe,PercentTradesNeverDelq_woe,MSinceMostRecentDelq_woe,MaxDelq2PublicRecLast12M_woe,...,PercentInstallTrades_woe,MSinceMostRecentInqexcl7days_woe,NumInqLast6M_woe,NumInqLast6Mexcl7days_woe,NetFractionRevolvingBurden_woe,NetFractionInstallBurden_woe,NumRevolvingTradesWBalance_woe,NumInstallTradesWBalance_woe,NumBank2NatlTradesWHighUtilization_woe,PercentTradesWBalance_woe
0,1.212731,0.026363,0.068578,-0.308196,-0.176618,0.982534,-0.179943,0.980232,1.145268,0.959119,...,0.077839,0.431781,-0.336157,-0.293976,0.204033,-0.212588,0.638483,-0.083471,0.254295,0.133993
1,1.212731,1.022666,-0.132759,0.793657,0.675509,0.982534,0.818584,-0.459261,-0.478819,0.283709,...,0.903738,0.431781,-0.336157,-0.293976,-0.723471,-0.212588,-0.017700,-0.415595,-0.423027,-0.834669
2,0.701260,1.022666,0.068578,0.793657,0.432487,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,0.077839,0.431781,0.425731,0.477439,0.429050,0.193523,-0.147205,-0.043969,0.254295,0.641411
3,0.701260,0.026363,0.107014,-0.051872,-0.176618,0.515046,0.634442,0.202690,-0.371713,0.024784,...,0.586148,0.431781,1.011981,0.477439,1.235124,0.226612,0.173665,0.324450,0.760856,1.109830
4,-1.352153,-0.669376,-0.357437,-0.639523,-0.012768,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,-0.136823,0.431781,0.038934,0.011598,0.429050,0.226612,-0.147205,-0.083471,-0.423027,0.361089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10454,0.019411,0.526229,0.068578,0.462762,-0.176618,-0.266698,-0.179943,0.202690,-0.371713,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,0.173665,-0.043969,-0.423027,1.109830
10455,0.701260,0.026363,-0.357437,-0.051872,0.272780,-0.266698,-0.179943,0.202690,0.089215,0.024784,...,0.047152,-0.044198,0.038934,0.011598,1.495328,-0.048350,-0.390585,-0.043969,0.254295,0.361089
10456,0.019411,0.526229,0.068578,0.462762,-0.012768,0.515046,0.634442,-0.459261,-0.478819,0.024784,...,-0.136823,-0.456448,0.425731,0.477439,-0.723471,-0.212588,0.173665,-0.415595,-0.423027,-0.347168
10457,0.019411,-0.217391,-0.132759,-0.639523,-0.365827,0.802181,0.818584,-0.459261,0.089215,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,-0.147205,-0.083471,-0.423027,-0.834669


In [67]:
df.head()

,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,...,PercentInstallTrades_woe,MSinceMostRecentInqexcl7days_woe,NumInqLast6M_woe,NumInqLast6Mexcl7days_woe,NetFractionRevolvingBurden_woe,NetFractionInstallBurden_woe,NumRevolvingTradesWBalance_woe,NumInstallTradesWBalance_woe,NumBank2NatlTradesWHighUtilization_woe,PercentTradesWBalance_woe
0,Bad,55,144,4,84,20,3,0,83,2,...,0.077839,0.431781,-0.336157,-0.293976,0.204033,-0.212588,0.638483,-0.083471,0.254295,0.133993
1,Bad,61,58,15,41,2,4,4,100,-7,...,0.903738,0.431781,-0.336157,-0.293976,-0.723471,-0.212588,-0.017700,-0.415595,-0.423027,-0.834669
2,Bad,67,66,5,24,9,0,0,100,-7,...,0.077839,0.431781,0.425731,0.477439,0.429050,0.193523,-0.147205,-0.043969,0.254295,0.641411
3,Bad,66,169,1,73,28,1,1,93,76,...,0.586148,0.431781,1.011981,0.477439,1.235124,0.226612,0.173665,0.324450,0.760856,1.109830
4,Bad,81,333,27,132,12,0,0,100,-7,...,-0.136823,0.431781,0.038934,0.011598,0.429050,0.226612,-0.147205,-0.083471,-0.423027,0.361089


In [53]:
optb = am.bp.get_binned_variable('PercentTradesWBalance')

In [54]:
# 2. Get the binning table object
binning_table = optb.binning_table

# 3. Build and display as a Pandas DataFrame
df_table = binning_table.build()
print(df_table)

                   Bin  Count  Count (%)  Non-event  Event  Event rate  \
0        (-inf, 44.50)   2376   0.227173        887   1489    0.626684   
1       [44.50, 60.50)   2187   0.209102        938   1249    0.571102   
2       [60.50, 71.50)   1737   0.166077        877    860    0.495107   
3       [71.50, 80.50)   1433   0.137011        829    604    0.421493   
4       [80.50, 87.50)    797   0.076202        514    283    0.355082   
5         [87.50, inf)   1929   0.184434       1414    515    0.266978   
6              Special      0   0.000000          0      0    0.000000   
7              Missing      0   0.000000          0      0    0.000000   
Totals                  10459   1.000000       5459   5000    0.478057   

             WoE        IV        JS  
0      -0.605843  0.081980  0.010094  
1      -0.374176  0.029176  0.003626  
2      -0.068253  0.000775  0.000097  
3       0.228818  0.007107  0.000886  
4       0.508949  0.019114  0.002364  
5       0.922183  0.14388